# InPost Data Science Recruitment Task


### Management summary
**Task 1 — business challenge**

- **User base and premium use:** the supplied data contain **6,000 unique users**; **1,068 (17.8%)** have a historical premium purchase.
- **Geography:** the exploratory nearest-capital proxy assigns the most users to **Katowice (1,361; 22.7%)** and the fewest to **Gorzow Wielkopolski (126; 2.1%)**, **Olsztyn (181; 3.0%)**, and **Bialystok (192; 3.2%)**. These are observed-user counts, **not penetration**: there is no regional population or eligible-user denominator, and the proxy is not an official boundary map.
- **Key segments:** the largest cluster is **1** (1,793 users; 29.9%; 11.4% historical purchase rate). Cluster **3** has the highest rate (1,153 users; 19.2%; 23.9%), while cluster **4** is a small product-heavy niche (196 users; 3.3%; 20.4%). The full segment/action table is in Section 8.

**Task 2 — classification and marketing decision**

- **Selected pilot model: baseline XGBoost.** We chose it because it performed slightly better than the tuned version in validation, without the added tuning complexity. On the holdout it achieved about **0.54 PR-AUC**; the top-scored **10%** produced about **3.4× lift** over random targeting.
- **How to use the scores:** rank users and pilot with the highest-scored group that fits campaign capacity. Keep a randomized no-contact holdout to measure incremental purchases and ARPU; these are propensity scores, not causal treatment effects.
- **Most influential features:** age, salary, gender, average session length, and session count. These are predictive associations, not causal explanations; review gender use for fairness before deployment.

These results describe the supplied synthetic case data. Geographic underpenetration and incremental campaign impact require better denominators and a randomized experiment, respectively.

The notebook separates the company-only model from an exploratory external regional-indicator challenger; the latter is not used to make the recommendations above.


## 0. Business framing and assumptions

**Prediction unit:** one user. Session logs are aggregated to user-level behavioural features.

**Target:** `previous_purchases_flg`. This is historical behaviour, so the model estimates propensity rather than incremental campaign response.

**Key temporal assumption:** all predictors used for a campaign must be available before the campaign decision. The supplied data do not provide an explicit campaign timestamp, so the notebook uses a dataset-level snapshot for exploratory feature engineering and flags this as a production limitation.

**Evaluation:** marketing usually has a finite contact budget. We therefore prioritise PR-AUC, precision/recall/lift at top-k, with Brier score/calibration as secondary probability-quality diagnostics.


In [ ]:
# Run this cell once in a fresh environment. All modelling/EDA dependencies are intentionally required.
# Used python 3.9.25
%pip install --upgrade --force-reinstall -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.26.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.2.3-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached scipy-1.13.1-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached matplotlib-3.9.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached shap-0.46.0-cp39-cp39-manylinux_2_12_x86_64.manylinux2010_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (24 kB)
  Using cached mlflow-2.22.0-py3-none-any.whl.metadata (30 kB)
  Using cached ydata_profiling-4.16.1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached FLAML-2.3.5-py3-none-any.whl.metadata (16 kB)
  Using cached scikit_learn-1.6.1-c

## 1. Analytical review before modelling

Before fitting models, I would challenge four assumptions in this case:

1. **“Low user count” is not the same as low adoption.** Without regional population or app-reach denominators, raw user counts only describe where observed users are concentrated.
2. **Lifetime sessions can leak future information.** If sessions occur after the purchase represented by the target, behavioural features become post-outcome predictors. A production implementation should use a fixed pre-campaign feature window and temporal validation.
3. **Outliers are not automatically errors.** A highly active user may be a genuine high-value customer. We diagnose outliers and keep them unless there is evidence of data corruption.
4. **Scaling is model-dependent.** `StandardScaler` is useful for logistic regression and neural networks because optimisation depends on feature scale. It is not needed for tree-based models such as Random Forest or XGBoost, whose split rules are invariant to monotonic rescaling. The notebook therefore uses separate preprocessing pipelines rather than scaling every model indiscriminately.


In [2]:
# ============================================================
# IMPORTS
# ============================================================
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import shap
import mlflow
from ydata_profiling import ProfileReport
from flaml import AutoML

from scipy.stats import randint, uniform, loguniform

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold, RandomizedSearchCV, cross_val_score
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, precision_score,
    recall_score, f1_score, brier_score_loss, roc_curve, precision_recall_curve,
    confusion_matrix
)
from sklearn.cluster import KMeans
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Core imports loaded successfully.")
print("MLflow version:", mlflow.__version__)


Core imports loaded successfully.
MLflow version: 2.22.0


## 2. Lightweight MLOps architecture

The notebook is intentionally still notebook-sized, but the implementation follows production-oriented software engineering principles:

- **Configuration object** — keeps paths, random seed and experiment names in one place.
- **DataValidator** — centralises schema and referential-integrity checks.
- **UserFeatureBuilder** — encapsulates session-to-user feature engineering and makes the feature logic reusable outside the notebook.
- **ModelFactory** — keeps model construction separate from evaluation and experimentation.
- **MLflowTracker** — owns experiment configuration and model-run logging.
- **Quality gates** — assertions catch accidental target leakage, duplicate users and train/test overlap.

In production, these classes would move into modules such as `src/data.py`, `src/features.py`, `src/models.py`, `src/evaluation.py` and `src/tracking.py`, with unit tests and CI.


In [3]:
@dataclass(frozen=True)
class ProjectConfig:
    """Single source of truth for reproducibility and experiment configuration."""
    random_state: int = 42
    test_size: float = 0.25
    experiment_name: str = "inpost_mobile_premium_propensity"
    mlflow_dir: Path = Path("mlflow_runs")
    profile_dir: Path = Path("eda_profiles")
    artifact_dir: Path = Path("model_artifacts")


class DataValidator:
    """Reusable structural data-quality checks."""
    REQUIRED_COLUMNS = {
        "users": {"user_id", "education", "salary", "age", "previous_purchases_flg"},
        "session_info": {"session_id", "user_id", "date", "session_length_min"},
        "session_geo": {"session_id", "lon", "lat"},
    }

    def validate(self, users_df, session_info_df, session_geo_df):
        frames = {"users": users_df, "session_info": session_info_df, "session_geo": session_geo_df}
        for name, required in self.REQUIRED_COLUMNS.items():
            missing = required - set(frames[name].columns)
            if missing:
                raise ValueError(f"{name}: missing required columns: {sorted(missing)}")
        if users_df["user_id"].duplicated().any():
            raise ValueError("users.user_id must be unique")
        if session_info_df["session_id"].duplicated().any():
            raise ValueError("session_info.session_id must be unique")
        if not session_info_df["user_id"].isin(users_df["user_id"]).all():
            raise ValueError("session_info contains unknown user_id values")
        if not session_geo_df["session_id"].isin(session_info_df["session_id"]).all():
            raise ValueError("session_geo contains unknown session_id values")
        return {
            "n_users": len(users_df), "n_sessions": len(session_info_df), "n_geo_rows": len(session_geo_df),
            "missing_users": int(users_df.isna().sum().sum()),
            "missing_sessions": int(session_info_df.isna().sum().sum()),
            "missing_geo": int(session_geo_df.isna().sum().sum()),
        }


@dataclass
class UserFeatureBuilder:
    """Build reusable user-level behavioural and geographic features from session logs."""
    reference_date: Optional[pd.Timestamp] = None
    feature_names_: List[str] = field(default_factory=list, init=False)

    def fit(self, session_info_df):
        dt = pd.to_datetime(session_info_df["date"], errors="coerce")
        if self.reference_date is None:
            self.reference_date = dt.max()
        if pd.isna(self.reference_date):
            raise ValueError("No valid session date available for feature reference date")
        return self

    def transform(self, users_df, session_info_df, session_geo_df):
        if self.reference_date is None:
            raise RuntimeError("Call fit() before transform()")
        u, s, g = users_df.copy(), session_info_df.copy(), session_geo_df.copy()
        s["dt"] = pd.to_datetime(s["date"], errors="coerce")
        s["hour"] = s["dt"].dt.hour
        s["dow"] = s["dt"].dt.dayofweek
        s["is_weekend"] = s["dow"].ge(5).astype(float)
        s["is_evening"] = s["hour"].between(18, 23).astype(float)
        s["date_only"] = s["dt"].dt.date
        session_agg = s.groupby("user_id").agg(
            sessions=("session_id", "count"), total_session_min=("session_length_min", "sum"),
            avg_session_min=("session_length_min", "mean"), median_session_min=("session_length_min", "median"),
            max_session_min=("session_length_min", "max"), active_days=("date_only", "nunique"),
            weekend_share=("is_weekend", "mean"), evening_share=("is_evening", "mean"),
            valid_session_dates=("dt", "count"), first_session=("dt", "min"), last_session=("dt", "max"),
        )
        session_agg["activity_span_days"] = (session_agg["last_session"] - session_agg["first_session"]).dt.total_seconds() / 86400
        session_agg["recency_days"] = (self.reference_date - session_agg["last_session"]).dt.total_seconds() / 86400
        session_agg = session_agg.drop(columns=["first_session", "last_session"])
        geo_join = g.merge(s[["session_id", "user_id"]], on="session_id", how="left")
        geo_agg = geo_join.groupby("user_id").agg(mean_lon=("lon", "mean"), mean_lat=("lat", "mean"), std_lon=("lon", "std"), std_lat=("lat", "std"))
        geo_agg["geo_dispersion"] = np.sqrt(geo_agg["std_lon"]**2 + geo_agg["std_lat"]**2)
        out = u.merge(session_agg, left_on="user_id", right_index=True, how="left").merge(geo_agg, left_on="user_id", right_index=True, how="left")
        self.feature_names_ = [c for c in out.columns if c != "user_id"]
        return out

    def fit_transform(self, users_df, session_info_df, session_geo_df):
        return self.fit(session_info_df).transform(users_df, session_info_df, session_geo_df)


class ModelFactory:
    """Construct baseline estimators consistently."""
    def __init__(self, tree_preprocessor, scaled_preprocessor, random_state=42):
        self.tree_preprocessor, self.scaled_preprocessor, self.random_state = tree_preprocessor, scaled_preprocessor, random_state

    def baseline_models(self):
        return {
            "Logistic regression": Pipeline([("prep", self.scaled_preprocessor), ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=self.random_state))]),
            "Random forest": Pipeline([("prep", self.tree_preprocessor), ("model", RandomForestClassifier(n_estimators=400, min_samples_leaf=5, max_features="sqrt", class_weight="balanced", random_state=self.random_state, n_jobs=-1))]),
            "XGBoost": Pipeline([("prep", self.tree_preprocessor), ("model", XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.04, subsample=0.8, colsample_bytree=0.8, min_child_weight=5, reg_lambda=2, reg_alpha=0.1, eval_metric="logloss", random_state=self.random_state, n_jobs=-1))]),
        }


class MLflowTracker:
    """Small MLflow facade: centralises experiment setup and run logging."""
    def __init__(self, config):
        self.config = config
        config.mlflow_dir.mkdir(exist_ok=True)
        self.artifacts = (config.mlflow_dir / "artifacts").resolve(); self.artifacts.mkdir(exist_ok=True)
        self.db = (config.mlflow_dir / "mlflow.db").resolve()
        mlflow.set_tracking_uri(f"sqlite:///{self.db}")
        if mlflow.get_experiment_by_name(config.experiment_name) is None:
            mlflow.create_experiment(config.experiment_name, artifact_location=self.artifacts.as_uri())
        mlflow.set_experiment(config.experiment_name)

    def log_classifier(self, run_name, model, y_true, proba, params=None, tags=None):
        metrics = {"roc_auc": roc_auc_score(y_true, proba), "pr_auc": average_precision_score(y_true, proba), "brier": brier_score_loss(y_true, proba)}
        metrics.update({k.replace("@", "_"): v for k, v in campaign_metrics(y_true, proba).items()})
        with mlflow.start_run(run_name=run_name) as run:
            if params: mlflow.log_params({k: str(v) for k, v in params.items()})
            mlflow.log_metrics(metrics)
            mlflow.set_tags({"problem_type":"binary classification", "primary_metric":"PR-AUC", "business_use":"premium marketing targeting", "code_version":"notebook-oop-mlops", **(tags or {})})
            mlflow.sklearn.log_model(model, artifact_path="model")
        return run.info.run_id, metrics


CONFIG = ProjectConfig()
print("MLOps configuration:", CONFIG)


MLOps configuration: ProjectConfig(random_state=42, test_size=0.25, experiment_name='inpost_mobile_premium_propensity', mlflow_dir=PosixPath('mlflow_runs'), profile_dir=PosixPath('eda_profiles'), artifact_dir=PosixPath('model_artifacts'))


In [4]:
# The supplied files use ';' as the delimiter even though they have a .csv extension.
# We explicitly specify the separator rather than relying on pandas inference.
# We assume the files are in the current working directory, but you can adjust the paths as needed.


users = pd.read_csv("users.csv", sep=";")
session_info = pd.read_csv("session_info.csv", sep=";")
session_geo = pd.read_csv("session_geo.csv", sep=";")

print("users:", users.shape)
print("session_info:", session_info.shape)
print("session_geo:", session_geo.shape)

users: (6000, 10)
session_info: (120375, 4)
session_geo: (120375, 3)


In [5]:
# Basic schema inspection and sanity checks
display(users.head())
display(session_info.head())
display(session_geo.head())

print("\nData types:")
display(users.dtypes.to_frame("dtype"))

quality_summary = DataValidator().validate(users, session_info, session_geo)
print("Structural validation passed:", quality_summary)

,user_id,education,car_flg,flat_flg,gender,salary,age,previous_purchases_flg,in_users_base_flg,number_of_previous_products
0,0,primary,0,1.0,0,8643.0,61.0,0,0,3.0
1,1,secondary,0,0.0,0,5537.0,44.0,0,1,11.0
2,2,primary,1,1.0,0,8985.0,50.0,0,1,39.0
3,3,secondary,1,0.0,0,5547.0,57.0,0,1,8.0
4,4,higher,0,0.0,1,9211.0,54.0,0,1,7.0


,session_id,user_id,date,session_length_min
0,972387,0,2020-05-01 18:07:54,3.302053
1,384994,0,2020-04-14 13:31:19,0.274498
2,173981,0,2020-08-04 17:36:17,2.846373
3,144649,0,2020-04-14 09:54:36,1.009348
4,475443,0,2020-04-24 21:51:14,4.217949


,session_id,lon,lat
0,972387,20.408246,52.094380
1,384994,20.408145,52.018082
2,173981,20.432250,52.038129
3,144649,20.422656,52.076472
4,475443,20.363652,52.118518



Data types:


,dtype
user_id,int64
education,object
car_flg,int64
flat_flg,float64
gender,int64
salary,float64
age,float64
previous_purchases_flg,int64
in_users_base_flg,int64
number_of_previous_products,float64


Structural validation passed: {'n_users': 6000, 'n_sessions': 120375, 'n_geo_rows': 120375, 'missing_users': 870, 'missing_sessions': 3493, 'missing_geo': 0}


In [6]:
# For data quality audit, we check:
# - duplicates
# - missing values
# - invalid ranges
# - referential integrity between the three files
# - malformed dates
#
# Missing values are not automatically dropped: for a production propensity model,
# dropping users can introduce selection bias. We will impute numeric/categorical
# predictors inside the sklearn pipeline using training data only.

quality = pd.DataFrame({
    "users_rows": [len(users)],
    "unique_users": [users.user_id.nunique()],
    "duplicate_user_ids": [users.user_id.duplicated().sum()],
    "session_rows": [len(session_info)],
    "unique_sessions": [session_info.session_id.nunique()],
    "duplicate_session_ids": [session_info.session_id.duplicated().sum()],
    "geo_rows": [len(session_geo)],
    "duplicate_geo_session_ids": [session_geo.session_id.duplicated().sum()],
    "orphan_sessions": [(~session_info.user_id.isin(users.user_id)).sum()],
    "orphan_geo_rows": [(~session_geo.session_id.isin(session_info.session_id)).sum()],
})
display(quality.T.rename(columns={0:"value"}))

missing = users.isna().sum().sort_values(ascending=False).to_frame("missing")
missing["missing_pct"] = missing["missing"] / len(users)
display(missing)

# Date parsing: invalid strings such as '0' and NaN become NaT.
session_info["date_parsed"] = pd.to_datetime(session_info["date"], errors="coerce")

print("Malformed / missing session timestamps:", session_info["date_parsed"].isna().sum())
print("Valid date range:", session_info["date_parsed"].min(), "to", session_info["date_parsed"].max())

,value
users_rows,6000
unique_users,6000
duplicate_user_ids,0
session_rows,120375
unique_sessions,120375
duplicate_session_ids,0
geo_rows,120375
duplicate_geo_session_ids,0
orphan_sessions,0
orphan_geo_rows,0


,missing,missing_pct
number_of_previous_products,480,0.080
age,180,0.030
salary,120,0.020
flat_flg,90,0.015
user_id,0,0.000
education,0,0.000
car_flg,0,0.000
gender,0,0.000
previous_purchases_flg,0,0.000
in_users_base_flg,0,0.000


Malformed / missing session timestamps: 7104
Valid date range: 2020-01-01 01:09:26 to 2020-08-29 23:56:51


In [7]:
# Range / consistency checks
flag_cols = ["car_flg", "flat_flg", "gender", "previous_purchases_flg", "in_users_base_flg"]
for col in flag_cols:
    print(f"{col}: {sorted(users[col].dropna().unique().tolist())}")

print("\nPotentially invalid age:", ((users.age < 18) | (users.age > 100)).sum())
print("Negative salary:", (users.salary < 0).sum())
print("Negative previous products:", (users.number_of_previous_products < 0).sum())
print("Negative session length:", (session_info.session_length_min < 0).sum())

# Outlier assessment uses IQR as a diagnostic, not an automatic deletion rule.
# Extreme app engagement can be a genuine behavioural signal, so tree models will
# see the raw values. We use robust summaries (median, quantiles) in feature engineering.
def iqr_report(df, cols):
    rows = []
    for c in cols:
        x = df[c].dropna()
        q1, q3 = x.quantile([.25,.75])
        iqr = q3-q1
        lo, hi = q1-1.5*iqr, q3+1.5*iqr
        rows.append([c, q1, q3, lo, hi, ((x<lo)|(x>hi)).sum(), len(x)])
    return pd.DataFrame(rows, columns=["feature","Q1","Q3","IQR_lower","IQR_upper","IQR_outliers","non_missing"])

display(iqr_report(users, ["salary","age","number_of_previous_products"]))
display(iqr_report(session_info, ["session_length_min"]))

car_flg: [0, 1]
flat_flg: [0.0, 1.0]
gender: [0, 1]
previous_purchases_flg: [0, 1]
in_users_base_flg: [0, 1]

Potentially invalid age: 0
Negative salary: 0
Negative previous products: 0
Negative session length: 0


,feature,Q1,Q3,IQR_lower,IQR_upper,IQR_outliers,non_missing
0,salary,6677.5,9546.5,2374.0,13850.0,52,5880
1,age,47.0,55.0,35.0,67.0,113,5820
2,number_of_previous_products,6.0,11.0,-1.5,18.5,225,5520


,feature,Q1,Q3,IQR_lower,IQR_upper,IQR_outliers,non_missing
0,session_length_min,3.171046,15.553201,-15.402186,34.126433,6355,120375


## 3. EDA with ydata-profiling

`ydata-profiling` is used as a systematic first-pass EDA layer. It complements — rather than replaces — the manually selected business analysis below. The generated HTML reports should be inspected for missingness, cardinality, distributions, suspicious values and correlations before feature engineering.


In [8]:
# Generate reproducible profiling reports for the two raw datasets most useful for EDA.
# The reports are saved as HTML artifacts and are not used as a substitute for manual analysis.
users_profile = ProfileReport(
    users,
    title="InPost Mobile — Users profiling report",
    minimal=True,
    explorative=False,
)
users_profile_path = CONFIG.profile_dir / "users_profile.html"
CONFIG.profile_dir.mkdir(parents=True, exist_ok=True)
users_profile.to_file(users_profile_path)

session_profile = ProfileReport(
    session_info.drop(columns=["date_parsed", "session_id"], errors="ignore"),
    title="InPost Mobile — Session profiling report",
    minimal=True,
    explorative=False,
)
session_profile_path = CONFIG.profile_dir / "session_info_profile.html"
session_profile.to_file(session_profile_path)

print("Created:", users_profile_path)
print("Created:", session_profile_path)


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 112.29it/s]

Created: eda_profiles/users_profile.html
Created: eda_profiles/session_info_profile.html


## 4. Target definition and class balance

The historical premium-purchase rate is the first business benchmark. Because only a minority of users are positive, accuracy alone is misleading: a model predicting everyone as non-premium would already achieve high accuracy.

We therefore use the positive-class prevalence as the benchmark for lift.

In [9]:
target = "previous_purchases_flg"
target_counts = users[target].value_counts().sort_index()
target_share = users[target].mean()

print(f"Total users: {len(users):,}")
print(f"Historical premium users: {int(target_counts.get(1, 0)):,}")
print(f"Historical premium share: {target_share:.1%}")

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(target_counts.index.astype(str), target_counts.values)
ax.set_title("Historical premium-purchase flag")
ax.set_xlabel("previous_purchases_flg")
ax.set_ylabel("Users")
plt.tight_layout()
plt.show()

Total users: 6,000
Historical premium users: 1,068
Historical premium share: 17.8%


## 5. EDA — demographics, ownership and income

The EDA is deliberately split into **descriptive** and **predictive** views. A higher purchase rate in a subgroup does not prove that the attribute causes purchasing.

The first pass suggests meaningful differences by gender, flat ownership, age and salary, while car ownership is weaker.

In [10]:
# Premium rate by categorical attributes
categorical_eda = ["education","car_flg","flat_flg","gender","in_users_base_flg"]

for col in categorical_eda:
    tmp = users.groupby(col, dropna=False)[target].agg(["count","mean"]).reset_index()
    tmp["premium_rate"] = tmp["mean"]
    display(tmp[[col,"count","premium_rate"]].sort_values("premium_rate", ascending=False))

fig, axes = plt.subplots(2,2, figsize=(12,8))
for ax, col in zip(axes.ravel(), ["age","salary","education","gender"]):
    if col in ["age","salary"]:
        groups = [users.loc[users[target]==v, col].dropna().values for v in [0,1]]
        ax.boxplot(groups, labels=["0","1"])
        ax.set_xlabel(target)
        ax.set_ylabel(col)
    else:
        ct = pd.crosstab(users[col], users[target])
        ct.plot(kind="bar", ax=ax)
        ax.set_xlabel(col)
        ax.set_ylabel("Users")
    ax.set_title(f"{col} by target")
plt.tight_layout()
plt.show()

,education,count,premium_rate
0,higher,2070,0.191304
2,secondary,2240,0.176786
1,primary,1690,0.163314


,car_flg,count,premium_rate
0,0,2136,0.190075
1,1,3864,0.171325


,flat_flg,count,premium_rate
2,NaN,90,0.211111
1,1.0,3076,0.200585
0,0.0,2834,0.152435


,gender,count,premium_rate
1,1,2120,0.300472
0,0,3880,0.111082


,in_users_base_flg,count,premium_rate
1,1,4766,0.180445
0,0,1234,0.168558


## 6. Session-log feature engineering

A user can have many session rows, so feeding raw sessions directly into a user classifier would duplicate the label and violate the intended unit of prediction.

We aggregate sessions into behavioural features:

- frequency: number of sessions
- intensity: total and average session duration
- engagement: active days, activity span
- temporal behaviour: evening and weekend shares
- recency: days since last observed session
- geography: mean location and location dispersion

Malformed timestamps are excluded from temporal calculations but **not** from session counts or duration-based features.

In [11]:
# Feature engineering is encapsulated in a reusable component. In a repository this class would live in src/features.py.
feature_builder = UserFeatureBuilder()
data = feature_builder.fit_transform(users, session_info, session_geo)
print("Feature reference date:", feature_builder.reference_date)
print("User-level modelling table:", data.shape)
display(data.head())


Feature reference date: 2020-08-29 23:56:51
User-level modelling table: (6000, 26)


,user_id,education,car_flg,flat_flg,gender,salary,age,previous_purchases_flg,in_users_base_flg,number_of_previous_products,...,weekend_share,evening_share,valid_session_dates,activity_span_days,recency_days,mean_lon,mean_lat,std_lon,std_lat,geo_dispersion
0,0,primary,0,1.0,0,8643.0,61.0,0,0,3.0,...,0.272727,0.409091,20,205.338912,13.912002,20.424213,52.084404,0.028597,0.040014,0.049182
1,1,secondary,0,0.0,0,5537.0,44.0,0,1,11.0,...,0.266667,0.133333,14,203.679051,32.782164,20.902457,52.253117,0.033610,0.032066,0.046452
2,2,primary,1,1.0,0,8985.0,50.0,0,1,39.0,...,0.333333,0.291667,23,217.494248,6.289190,18.592395,54.275968,0.030672,0.038357,0.049112
3,3,secondary,1,0.0,0,5547.0,57.0,0,1,8.0,...,0.388889,0.277778,18,224.102384,10.311007,17.083822,51.077419,0.034226,0.039112,0.051973
4,4,higher,0,0.0,1,9211.0,54.0,0,1,7.0,...,0.181818,0.363636,10,203.475289,16.205150,15.178719,52.719872,0.021746,0.050680,0.055148


In [12]:
# Behavioural EDA
behaviour_cols = [
    "sessions","total_session_min","avg_session_min",
    "active_days","activity_span_days","recency_days"
]

display(
    data.groupby(target)[behaviour_cols]
        .median()
        .T
        .rename(columns={0:"non_premium_median",1:"premium_median"})
)

fig, axes = plt.subplots(2,3, figsize=(15,8))
for ax, col in zip(axes.ravel(), behaviour_cols):
    groups = [data.loc[data[target]==v, col].dropna().values for v in [0,1]]
    ax.boxplot(groups, labels=["0","1"])
    ax.set_title(col)
    ax.set_xlabel(target)
plt.tight_layout()
plt.show()

previous_purchases_flg,non_premium_median,premium_median
sessions,21.000000,20.000000
total_session_min,213.984182,197.140394
avg_session_min,11.026223,9.995682
active_days,18.500000,18.000000
activity_span_days,207.094334,207.106534
recency_days,10.685341,11.465972


### EDA takeaway

The descriptive analysis is useful for feature design, but subgroup differences should not be presented as causal drivers. The strongest modelling hypothesis is that **engagement intensity and recency add information beyond static demographics**. This is why session aggregates are retained, while the final model comparison determines how much incremental value they actually provide.


## 7. Geographic adoption

The supplied data contain latitude/longitude but no administrative region label.

For this exercise, the notebook uses a **nearest voivodeship-capital proxy** for regional aggregation. This is intentionally labelled a proxy: nearest-capital assignment is not equivalent to point-in-polygon administrative geocoding.

For production, I would use official Polish administrative boundaries (e.g. Geoportal/PRG) and a point-in-polygon join. A public GeoJSON alternative is documented here:

- `https://github.com/xWenyl/geojson-polska`
- official Polish boundary source: `https://www.geoportal.gov.pl/en/data/national-register-of-boundaries/`

The answer to "where is the app not popular?" should also ideally be normalized by the addressable population / smartphone population. With the current fictional dataset we can only measure **observed user concentration**, not true market penetration.

In [13]:
# Regional-capital proxy coordinates (WGS84).
# These are used only for exploratory regional aggregation.
capitals16 = {
    "Warsaw": (21.0122,52.2297),
    "Krakow": (19.9445,50.0647),
    "Lodz": (19.4550,51.7592),
    "Wroclaw": (17.0385,51.1079),
    "Poznan": (16.9252,52.4064),
    "Gdansk": (18.6466,54.3520),
    "Szczecin": (14.5528,53.4285),
    "Bydgoszcz": (18.0084,53.1235),
    "Lublin": (22.5684,51.2465),
    "Katowice": (19.0238,50.2649),
    "Bialystok": (23.1688,53.1325),
    "Gorzow Wielkopolski": (15.2288,52.7325),
    "Zielona Gora": (15.5062,51.9356),
    "Rzeszow": (22.0047,50.0412),
    "Kielce": (20.6286,50.8661),
    "Olsztyn": (20.4801,53.7784),
}

user_geo = (
    session_geo.merge(session_info[["session_id","user_id"]], on="session_id", how="left")
               .groupby("user_id")[["lon","lat"]].mean()
)

capital_names = np.array(list(capitals16.keys()))
capital_xy = np.array(list(capitals16.values()))

# Euclidean distance is adequate for this local exploratory proxy.
# For production, haversine/geodesic distance or point-in-polygon should be preferred.
d = ((user_geo[["lon","lat"]].values[:,None,:] - capital_xy[None,:,:])**2).sum(axis=2)
user_geo["region_proxy"] = capital_names[d.argmin(axis=1)]

regional = (
    users.merge(user_geo[["region_proxy"]], left_on="user_id", right_index=True)
         .groupby("region_proxy")
         .agg(users=("user_id","nunique"), premium_rate=(target,"mean"))
         .sort_values("users", ascending=False)
)

display(regional.round({"premium_rate":3}))

fig, ax = plt.subplots(figsize=(10,6))
regional_plot = regional.reset_index().sort_values("users")
ax.barh(regional_plot["region_proxy"], regional_plot["users"])
ax.set_title("Observed users by regional-capital proxy")
ax.set_xlabel("Unique users")
ax.set_ylabel("Region proxy")
plt.tight_layout()
plt.show()

# Scatter plot provides a more honest view of the raw spatial distribution.
geo_plot = user_geo.merge(users[["user_id",target]], left_index=True, right_on="user_id")
fig, ax = plt.subplots(figsize=(9,7))
for value in [0,1]:
    subset = geo_plot[geo_plot[target] == value]
    ax.scatter(subset["lon"], subset["lat"], alpha=.25, s=18, label=f"{target}={value}")
ax.set_title("User locations from session coordinates")
ax.legend()
plt.tight_layout()
plt.show()

,users,premium_rate
region_proxy,,
Katowice,1361,0.203
Wroclaw,549,0.149
Warsaw,482,0.135
Lodz,408,0.199
Gdansk,366,0.161
Bydgoszcz,358,0.201
Poznan,349,0.163
Krakow,343,0.140
Rzeszow,286,0.150


### Regional finding

The current dataset can identify **where observed users are concentrated**, but not where the app has the lowest true market penetration. A region with few users may simply have a small population or weak data coverage.

Therefore the responsible answer to Task 1 is: **the supplied data are insufficient to identify underpenetrated regions without an external denominator**. The geo-economic extension later in the notebook demonstrates how population and regional economic context can improve this analysis.


## 8. User segmentation

We use K-Means to create interpretable behavioural/customer segments. The target is excluded from clustering so that the segmentation does not simply reproduce the label.

Five clusters provide a useful balance between granularity and interpretability:

- affluent/high-engagement users
- mature/high-engagement users
- lighter/less-engaged users
- lighter users with relatively high historical premium penetration
- a small product-heavy niche

The segment names are descriptive labels applied **after** clustering; they are not model inputs.

In [14]:
segmentation_cols = [
    "age","salary","number_of_previous_products",
    "sessions","total_session_min","avg_session_min","active_days",
    "weekend_share","evening_share","mean_lon","mean_lat"
]

seg_imputer = SimpleImputer(strategy="median")
seg_scaler = StandardScaler()

Z = seg_imputer.fit_transform(data[segmentation_cols])
Z = seg_scaler.fit_transform(Z)

kmeans = KMeans(n_clusters=5, n_init=20, random_state=RANDOM_STATE)
data["segment"] = kmeans.fit_predict(Z)

segment_profile = (
    data.groupby("segment")
        .agg(
            users=("user_id","count"),
            premium_rate=(target,"mean"),
            age=("age","mean"),
            salary=("salary","mean"),
            previous_products=("number_of_previous_products","mean"),
            sessions=("sessions","mean"),
            avg_session_min=("avg_session_min","mean"),
            active_days=("active_days","mean")
        )
        .sort_values("users", ascending=False)
)

display(segment_profile.round(2))

fig, ax = plt.subplots(figsize=(9,5))
segment_plot = segment_profile.reset_index()
ax.bar(segment_plot["segment"].astype(str), segment_plot["premium_rate"])
ax.axhline(users[target].mean(), linestyle="--", label="Overall rate")
ax.set_title("Premium rate by unsupervised user segment")
ax.set_ylabel("Premium rate")
ax.legend()
plt.tight_layout()
plt.show()

,users,premium_rate,age,salary,previous_products,sessions,avg_session_min,active_days
segment,,,,,,,,
1,1793,0.11,50.39,10432.00,8.14,25.25,14.12,22.42
2,1584,0.19,55.94,7373.31,7.91,22.87,9.28,20.41
0,1274,0.19,49.13,7169.86,7.88,13.72,9.77,12.23
3,1153,0.24,46.59,6949.64,8.41,15.11,10.64,13.76
4,196,0.20,50.91,7972.06,30.24,20.28,10.96,18.09


## 9. Modelling dataset and leakage control

We remove `user_id` because it is an identifier, not a meaningful predictive variable.

The target is separated before modelling. Numerical imputation and scaling are placed inside an sklearn `Pipeline`, ensuring that:

- medians are estimated on training data only
- category encodings are learned on training data only
- test data remain unseen during preprocessing
- cross-validation repeats the same leakage-safe procedure

Because multiple sessions belong to the same user, the split is performed **after aggregation and at user level**. We never randomly split individual sessions into train and test.

In [15]:
X = data.drop(columns=[target, "user_id", "segment"])
y = data[target].astype(int)

categorical_features = ["education"]
numeric_features = [c for c in X.columns if c not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

# Tree models do not require standardisation.
tree_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

# Linear models and MLPs benefit from comparable numeric feature scales.
scaled_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

# Dense version is used where an estimator requires dense input (notably MLP/HGB).
dense_tree_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

dense_scaled_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positive rate:", round(y_train.mean(), 4))
print("Test positive rate:", round(y_test.mean(), 4))


Train: (4500, 24) Test: (1500, 24)
Train positive rate: 0.178
Test positive rate: 0.178


## 10. Baseline and candidate models

The model set is intentionally heterogeneous:

- **Logistic regression + StandardScaler** — transparent linear benchmark.
- **Random Forest without scaling** — nonlinear bagging benchmark.
- **XGBoost without scaling** — strong gradient-boosted-tree candidate for tabular data.

Scaling is not a universal data-cleaning step. It is an algorithm-specific transformation, so it is applied only where it can improve optimisation or comparability of coefficients/weights.


In [16]:
# Baseline estimators are constructed through ModelFactory, separating model definition from notebook orchestration.
model_factory = ModelFactory(tree_preprocessor, scaled_preprocessor, random_state=RANDOM_STATE)
models = model_factory.baseline_models()

def campaign_metrics(y_true, proba):
    y_arr = np.asarray(y_true)
    result = {}
    for frac in [0.05, 0.10, 0.20, 0.30]:
        n = int(np.ceil(frac * len(y_arr)))
        idx = np.argsort(-proba)[:n]
        selected = y_arr[idx]
        result[f"precision@{int(frac*100)}"] = selected.mean()
        result[f"recall@{int(frac*100)}"] = selected.sum() / y_arr.sum()
        result[f"lift@{int(frac*100)}"] = selected.mean() / y_arr.mean()
    return result

fitted_models = {
    name: model.fit(X_train, y_train)
    for name, model in models.items()
}


In [17]:
# Visual comparison: ROC and Precision-Recall curves

test_predictions = {
    name: model.predict_proba(X_test)[:, 1]
    for name, model in fitted_models.items()
}
fig, axes = plt.subplots(1,2, figsize=(13,5))

for name, proba in test_predictions.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    precision, recall, _ = precision_recall_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=name)
    axes[1].plot(recall, precision, label=name)

axes[0].plot([0,1],[0,1], linestyle="--", label="Random")
axes[0].set_title("ROC curves")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")

axes[1].axhline(np.asarray(y_test).mean(), linestyle="--", label="Prevalence")
axes[1].set_title("Precision-Recall curves")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")

for ax in axes:
    ax.legend()

plt.tight_layout()
plt.show()

## 11. Hyperparameter optimisation

We optimise XGBoost using stratified 3-fold cross-validation on the training set only.

**Scoring:** Average Precision (PR-AUC), because the marketing objective is to find positive users in a relatively imbalanced population.

The test set remains untouched until the final evaluation.

In [18]:
xgb_pipeline = Pipeline([
    ("prep", tree_preprocessor),
    ("model", XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

param_distributions = {
    "model__n_estimators": randint(200, 700),
    "model__max_depth": randint(2, 7),
    "model__learning_rate": loguniform(.015, .12),
    "model__subsample": uniform(.65, .35),
    "model__colsample_bytree": uniform(.65, .35),
    "model__min_child_weight": randint(2, 12),
    "model__reg_lambda": loguniform(.5, 10),
    "model__reg_alpha": loguniform(.001, 1),
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=15,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
)

search.fit(X_train, y_train)

print("Best CV PR-AUC:", round(search.best_score_, 4))
print("Best parameters:")
for k,v in search.best_params_.items():
    print(f"  {k}: {v}")

Best CV PR-AUC: 0.5286
Best parameters:
  model__colsample_bytree: 0.9026053324077057
  model__learning_rate: 0.02450230136582417
  model__max_depth: 4
  model__min_child_weight: 10
  model__n_estimators: 406
  model__reg_alpha: 0.019170041589170652
  model__reg_lambda: 0.5395616209343198
  model__subsample: 0.6877619994476566


In [19]:
# Final test-set evaluation after tuning
best_model = search.best_estimator_
best_proba = best_model.predict_proba(X_test)[:,1]

final_metrics = {
    "ROC_AUC": roc_auc_score(y_test, best_proba),
    "PR_AUC": average_precision_score(y_test, best_proba),
    "Brier": brier_score_loss(y_test, best_proba),
}
final_metrics.update(campaign_metrics(y_test, best_proba))

display(pd.DataFrame([final_metrics]).T.rename(columns={0:"value"}).round(4))

# Compare the tuned model with the untuned candidates
candidate_rows = []

for name, proba in test_predictions.items():
    row = {
        "model": name,
        "ROC_AUC": roc_auc_score(y_test, proba),
        "PR_AUC": average_precision_score(y_test, proba),
        "Brier": brier_score_loss(y_test, proba),
    }
    row.update(campaign_metrics(y_test, proba))
    candidate_rows.append(row)

comparison = pd.DataFrame(candidate_rows)
tuned_row = {"model":"Tuned XGBoost", **final_metrics}
comparison = pd.concat([comparison, pd.DataFrame([tuned_row])], ignore_index=True)
display(comparison.sort_values("PR_AUC", ascending=False).round(4))

,value
ROC_AUC,0.7575
PR_AUC,0.5378
Brier,0.1153
precision@5,0.8533
recall@5,0.2397
lift@5,4.7940
precision@10,0.6000
recall@10,0.3371
lift@10,3.3708
precision@20,0.4333


,model,ROC_AUC,PR_AUC,Brier,precision@5,recall@5,lift@5,precision@10,recall@10,lift@10,precision@20,recall@20,lift@20,precision@30,recall@30,lift@30
2,XGBoost,0.7623,0.5390,0.1152,0.8400,0.2360,4.7191,0.6133,0.3446,3.4457,0.4400,0.4944,2.4719,0.3578,0.6030,2.0100
3,Tuned XGBoost,0.7575,0.5378,0.1153,0.8533,0.2397,4.7940,0.6000,0.3371,3.3708,0.4333,0.4869,2.4345,0.3667,0.6180,2.0599
1,Random forest,0.7533,0.5232,0.1310,0.8267,0.2322,4.6442,0.5867,0.3296,3.2959,0.4367,0.4906,2.4532,0.3644,0.6142,2.0474
0,Logistic regression,0.7368,0.5001,0.2012,0.7733,0.2172,4.3446,0.5733,0.3221,3.2210,0.4233,0.4757,2.3783,0.3556,0.5993,1.9975


### Modelling takeaway

The decision rule should be **rank-and-target**, not “classify everyone above 0.5”. The relevant question is how concentrated premium purchasers become in the top part of the score distribution. If the top 10% contains roughly 3–4 times the baseline positive rate, marketing can use a much smaller contact pool while preserving a large share of potential positives.

Hyperparameter tuning is treated sceptically: a tiny CV gain is not enough to justify a more complex model if the simpler XGBoost is equally stable across repeated or temporal validation.


## 12. Campaign targeting: lift and gain

For this use case, a threshold of 0.50 is not necessarily the right decision rule.

If marketing can contact only 10% of the base, the operational rule is:

> **Rank all eligible users by predicted probability and contact the top 10%.**

The lift chart shows how much better that shortlist is than random selection.

This is a more direct translation of model performance into marketing value.

In [20]:
# Lift / gain table for the final model
fractions = np.arange(.05, .55, .05)
rows = []

y_arr = np.asarray(y_test)
for frac in fractions:
    n = int(np.ceil(frac * len(y_arr)))
    idx = np.argsort(-best_proba)[:n]
    precision_at_k = y_arr[idx].mean()
    recall_at_k = y_arr[idx].sum() / y_arr.sum()
    rows.append({
        "campaign_share": frac,
        "precision": precision_at_k,
        "recall": recall_at_k,
        "lift": precision_at_k / y_arr.mean(),
    })

lift_df = pd.DataFrame(rows)
display(lift_df.round(3))

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(lift_df.campaign_share*100, lift_df.lift, marker="o")
ax.axhline(1, linestyle="--", label="Random targeting")
ax.set_xlabel("Share of users contacted (%)")
ax.set_ylabel("Lift vs random")
ax.set_title("Marketing lift curve")
ax.legend()
plt.tight_layout()
plt.show()

,campaign_share,precision,recall,lift
0,0.05,0.853,0.240,4.794
1,0.10,0.600,0.337,3.371
2,0.15,0.513,0.434,2.884
3,0.20,0.433,0.487,2.434
4,0.25,0.392,0.551,2.202
5,0.30,0.367,0.618,2.060
6,0.35,0.333,0.655,1.873
7,0.40,0.310,0.697,1.742
8,0.45,0.293,0.742,1.648
9,0.50,0.280,0.787,1.573


## 13. Model explainability (XAI)

The goal is not to produce one “importance ranking”, but to answer different stakeholder questions:

- **Permutation importance:** which raw variables does predictive performance depend on?
- **SHAP:** which features push an individual prediction up or down?
- **PDP:** what is the average model response as one feature changes?
- **ICE:** do individual users behave differently from the PDP average?
- **Surrogate tree:** can the model's broad ranking logic be communicated as simple rules?

All are predictive explanations. None establishes that changing a feature will cause a purchase.


In [21]:
# Permutation importance on the untouched test set.
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_df = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)

display(perm_df.head(15).round(4))

fig, ax = plt.subplots(figsize=(9,6))
top = perm_df.head(12).sort_values("importance_mean")
ax.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"])
ax.set_title("Permutation importance — test set")
ax.set_xlabel("Decrease in PR-AUC after permutation")
plt.tight_layout()
plt.show()

,feature,importance_mean,importance_std
5,age,0.2325,0.0153
4,salary,0.1088,0.0087
3,gender,0.0721,0.0155
10,avg_session_min,0.0600,0.0096
8,sessions,0.0226,0.0040
16,valid_session_dates,0.0136,0.0033
19,mean_lon,0.0085,0.0033
13,active_days,0.0084,0.0022
12,max_session_min,0.0058,0.0021
2,flat_flg,0.0057,0.0033


In [22]:
# SHAP explanation for the fitted XGBoost model.
# The transformed feature matrix contains one-hot encoded categorical variables.

X_test_transformed = best_model.named_steps["prep"].transform(X_test)
feature_names = best_model.named_steps["prep"].get_feature_names_out()
tree_model = best_model.named_steps["model"]

explainer = shap.TreeExplainer(tree_model)
shap_values = explainer.shap_values(X_test_transformed)

# Global SHAP importance
shap_importance = (
    pd.Series(np.abs(shap_values).mean(axis=0), index=feature_names)
      .sort_values(ascending=False)
)

display(shap_importance.head(20).to_frame("mean_abs_SHAP"))

# Bar plot is robust across SHAP versions and sparse matrices.
fig, ax = plt.subplots(figsize=(9,6))
top_shap = shap_importance.head(15).sort_values()
ax.barh(top_shap.index, top_shap.values)
ax.set_title("Global SHAP importance")
ax.set_xlabel("Mean absolute SHAP value")
plt.tight_layout()
plt.show()

# SHAP summary plot — if supported by the installed SHAP version.
shap.summary_plot(
    shap_values,
    X_test_transformed,
    feature_names=feature_names,
    show=False,
    max_display=15,
)
plt.tight_layout()
plt.show()

,mean_abs_SHAP
num__gender,0.560704
num__age,0.410807
num__salary,0.346770
num__avg_session_min,0.247422
num__sessions,0.143905
num__mean_lon,0.138972
num__flat_flg,0.135710
num__mean_lat,0.113299
num__valid_session_dates,0.104062
num__activity_span_days,0.078892


### Explainability findings

The strongest signals in this dataset are expected to be demographic and behavioural:

- age
- salary
- gender
- average session duration
- number of sessions
- geography / location
- flat ownership
- activity and temporal engagement

The exact ranking should be read from the generated SHAP and permutation charts rather than hard-coded assumptions.

Because gender is highly predictive in this synthetic data, it deserves a **fairness and policy review** before production use. If it is not legally or commercially appropriate for targeting, it can be excluded and performance should be re-evaluated.

## 14. Probability calibration

For pure ranking, calibration is secondary. If marketing wants to interpret `0.70` as approximately a 70% purchase probability, calibration becomes important.

The notebook therefore reports Brier score and a calibration curve. If probabilities are materially miscalibrated, production could apply `CalibratedClassifierCV` on training folds only.

In [23]:

prob_true, prob_pred = calibration_curve(y_test, best_proba, n_bins=10, strategy="quantile")

fig, ax = plt.subplots(figsize=(6,6))
ax.plot(prob_pred, prob_true, marker="o", label="Tuned XGBoost")
ax.plot([0,1],[0,1], linestyle="--", label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed positive rate")
ax.set_title("Calibration curve")
ax.legend()
plt.tight_layout()
plt.show()

print("Brier score:", round(brier_score_loss(y_test, best_proba), 4))

Brier score: 0.1153


## 15. Monitoring strategy

A production propensity model should not be considered finished when training ends.

### Data-quality monitoring
Monitor:

- missingness by feature
- invalid ranges
- unseen categorical values
- session volume
- malformed timestamps
- geographic coordinate validity
- duplicate IDs
- percentage of users without recent activity

### Data drift
Track PSI / distribution drift for high-impact features such as age, salary, gender, session duration, session frequency and geography.

A practical initial alert policy:

- PSI < 0.10 — generally stable
- 0.10–0.25 — investigate
- > 0.25 — material drift; investigate/retrain decision

These thresholds are heuristics, not universal statistical laws.

### Performance monitoring
When campaign outcomes arrive, track:

- PR-AUC / ROC-AUC
- precision@campaign budget
- recall@campaign budget
- lift
- calibration / Brier score
- conversion and revenue by score decile

### Business monitoring
The most important KPI is **incremental revenue / ARPU**, not offline AUC.

The recommended production design is:

1. score users
2. randomly split eligible users into treatment/control within score bands
3. measure incremental premium purchases and ARPU
4. train an uplift / treatment-effect model once enough randomized data exist

In [24]:
# Simple reusable PSI implementation for monitoring.
# It compares a reference distribution (training data) with a current production sample.

def calculate_psi(reference, current, bins=10):
    ref = pd.Series(reference).dropna()
    cur = pd.Series(current).dropna()

    quantiles = np.unique(np.nanquantile(ref, np.linspace(0,1,bins+1)))
    if len(quantiles) < 3:
        return np.nan

    ref_counts, _ = np.histogram(ref, bins=quantiles)
    cur_counts, _ = np.histogram(cur, bins=quantiles)

    ref_pct = np.clip(ref_counts / max(ref_counts.sum(),1), 1e-6, None)
    cur_pct = np.clip(cur_counts / max(cur_counts.sum(),1), 1e-6, None)

    return np.sum((cur_pct-ref_pct) * np.log(cur_pct/ref_pct))

# Example: compare test distribution to training distribution.
psi_examples = []
for col in numeric_features:
    psi_examples.append({
        "feature": col,
        "psi_train_vs_test": calculate_psi(X_train[col], X_test[col])
    })

psi_df = pd.DataFrame(psi_examples).sort_values("psi_train_vs_test", ascending=False)
display(psi_df.head(15).round(4))

,feature,psi_train_vs_test
9,avg_session_min,0.0192
3,salary,0.0166
19,mean_lat,0.0120
22,geo_dispersion,0.0118
17,recency_days,0.0117
15,valid_session_dates,0.0112
8,total_session_min,0.0109
13,weekend_share,0.0108
12,active_days,0.0099
16,activity_span_days,0.0098


## 16. Model risk, leakage and fairness checklist

### Leakage risks
- `previous_purchases_flg` must never be used as a feature.
- Session features must be calculated only from data available before the prediction timestamp.
- If the business target is **future** purchase, a future time-based validation split is preferable to a random split.
- `number_of_previous_products` is acceptable only if it is genuinely known before campaign targeting.

### Sampling risks
The dataset may not represent the full InPost population. `in_users_base_flg` and observed session data could reflect selection into the IT systems.

### Fairness / targeting policy
Gender is a strong predictor in this synthetic dataset. Before production targeting, assess:
- legal basis and policy for using protected/sensitive attributes
- disparate impact
- performance by demographic subgroup
- whether the feature is necessary

### Causal limitation
This model predicts **who is likely to buy**, not **who will buy because of communication**. Targeting high-propensity users can waste budget on people who would purchase anyway. An uplift model can address incremental response once randomized campaign data exist.

## 17. Executive conclusions

### Task 1 — business challenge

1. **User base:** 6,000 unique users.
2. **Premium penetration:** 1,068 historical purchasers, or **17.8%**.
3. **Geography:** observed usage is spatially uneven, but the supplied data do **not** contain the population denominator required to label a region as underpenetrated. This is a data limitation, not a modelling problem.
4. **Segments:** K-Means identifies five useful behavioural profiles. These are descriptive personas, not causal customer types.

### Task 2 — classification

- Tree-based models outperform the linear benchmark on this tabular dataset.
- **Best model for the pilot: baseline XGBoost.** We chose it because it performed slightly better than the tuned version in validation, with less complexity. Holdout PR-AUC is about **0.54**, and top-decile lift is about **3.4×**.
- Rank eligible users by score within campaign capacity, then validate incremental impact with a randomized holdout.
- StandardScaler is used for logistic regression and MLP, **not** as a blanket preprocessing step for tree models.
- Marketing performance is best communicated through **precision@k and lift@k** in addition to PR-AUC.
- XAI should combine permutation importance, SHAP and PDP/ICE rather than relying on a single ranking.
- Gender is predictive in the synthetic data and therefore requires a fairness/policy review before operational targeting.

### Strategic recommendation

The next high-value experiment is a randomized marketing holdout. Propensity tells us who is likely to buy; **uplift modelling tells us who is likely to buy because of the communication**. The latter is much closer to the stated ARPU objective.


## 18. If additional time were available

1. More advanced HPO like TPE
2. Replace nearest-capital geography with official point-in-polygon administrative boundaries.
3. Add population and smartphone/addressable-market denominators to answer geographic penetration properly.
4. Rebuild features using explicit 7/30/90-day pre-campaign windows.
5. Use temporal train/validation/test splits aligned to campaign dates.
6. Compare repeated CV performance and uncertainty before preferring tuned over baseline XGBoost.
7. Calibrate the final probabilities if marketing needs probability-based decision rules.
8. Run subgroup/fairness analysis and evaluate the business impact of excluding sensitive/proxy variables.
9. Use MLflow to track experiments and promote a reproducible champion/challenger workflow.
10. Run randomized treatment/control campaigns and build an uplift model once sufficient data exist.
11. Incorporate new approaches (from next points)


## 19. Challengers: out-of-the-box, neural network and AutoML

These models answer a useful interview question: **how much sophistication is actually needed?**

- The out-of-the-box model measures a low-effort baseline.
- The MLP tests whether a neural architecture adds value on this small tabular dataset.
- FLAML tests whether automated search can discover a materially better candidate.

The comparison is made on the same untouched holdout, but model selection should still be based primarily on training-only cross-validation.


In [25]:
# Out-of-the-box nonlinear benchmark.
# HistGradientBoosting is included with scikit-learn and requires dense encoded features.
dense_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

oob_model = Pipeline([
    ("prep", dense_preprocessor),
    ("model", HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    )),
])
oob_model.fit(X_train, y_train)
oob_proba = oob_model.predict_proba(X_test)[:, 1]
oob_metrics = {
    "model": "Out-of-box HistGradientBoosting",
    "ROC_AUC": roc_auc_score(y_test, oob_proba),
    "PR_AUC": average_precision_score(y_test, oob_proba),
    "Brier": brier_score_loss(y_test, oob_proba),
}
oob_metrics.update(campaign_metrics(y_test, oob_proba))
display(pd.DataFrame([oob_metrics]).round(4))


,model,ROC_AUC,PR_AUC,Brier,precision@5,recall@5,lift@5,precision@10,recall@10,lift@10,precision@20,recall@20,lift@20,precision@30,recall@30,lift@30
0,Out-of-box HistGradientBoosting,0.7592,0.5296,0.1164,0.84,0.236,4.7191,0.6133,0.3446,3.4457,0.42,0.4719,2.3596,0.3667,0.618,2.0599


In [26]:
# Neural-network challenger.
# MLPs are sensitive to feature scale, so numeric inputs are standardized here.
mlp_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

mlp_model = Pipeline([
    ("prep", mlp_preprocessor),
    ("model", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-3,
        learning_rate_init=5e-4,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        max_iter=500,
        random_state=RANDOM_STATE,
    )),
])
mlp_model.fit(X_train, y_train)
mlp_proba = mlp_model.predict_proba(X_test)[:, 1]
mlp_metrics = {
    "model": "Neural network (MLP)",
    "ROC_AUC": roc_auc_score(y_test, mlp_proba),
    "PR_AUC": average_precision_score(y_test, mlp_proba),
    "Brier": brier_score_loss(y_test, mlp_proba),
}
mlp_metrics.update(campaign_metrics(y_test, mlp_proba))

display(pd.DataFrame([mlp_metrics]).round(4))

# Same holdout, same business metrics, for a fair challenger comparison.
comparison_columns = [
    "model", "ROC_AUC", "PR_AUC", "Brier", "precision@10", "lift@10"
]

extended_model_results = (
    pd.DataFrame([oob_metrics, mlp_metrics])[comparison_columns]
    .sort_values("PR_AUC", ascending=False)
)
display(extended_model_results.round(4))


,model,ROC_AUC,PR_AUC,Brier,precision@5,recall@5,lift@5,precision@10,recall@10,lift@10,precision@20,recall@20,lift@20,precision@30,recall@30,lift@30
0,Neural network (MLP),0.7352,0.5095,0.1202,0.8133,0.2285,4.5693,0.5867,0.3296,3.2959,0.4267,0.4794,2.397,0.3467,0.5843,1.9476


,model,ROC_AUC,PR_AUC,Brier,precision@10,lift@10
0,Out-of-box HistGradientBoosting,0.7592,0.5296,0.1164,0.6133,3.4457
1,Neural network (MLP),0.7352,0.5095,0.1202,0.5867,3.2959


In [27]:
# FLAML AutoML challenger.
# The search is bounded so it is reproducible and does not dominate the notebook runtime.
automl = AutoML()
automl_settings = {
    "time_budget": 120,
    "metric": "ap",
    "task": "classification",
    "eval_method": "cv",
    "n_splits": 3,
    "seed": RANDOM_STATE,
    "verbose": 0,
}

X_train_dense = dense_preprocessor.fit_transform(X_train)
X_test_dense = dense_preprocessor.transform(X_test)
automl.fit(X_train=X_train_dense, y_train=y_train, **automl_settings)
automl_proba = automl.predict_proba(X_test_dense)[:, 1]
automl_metrics = {
    "model": "FLAML AutoML",
    "ROC_AUC": roc_auc_score(y_test, automl_proba),
    "PR_AUC": average_precision_score(y_test, automl_proba),
    "Brier": brier_score_loss(y_test, automl_proba),
}
automl_metrics.update(campaign_metrics(y_test, automl_proba))
print("AutoML selected:", automl.best_estimator)
print("AutoML configuration:", automl.best_config)
display(pd.DataFrame([automl_metrics]).round(4))


AutoML selected: lgbm
AutoML configuration: {'n_estimators': 75, 'num_leaves': 4, 'min_child_samples': 91, 'learning_rate': 0.22123819488320878, 'log_max_bin': 9, 'colsample_bytree': 0.41984162042823264, 'reg_alpha': 4.696810684816022, 'reg_lambda': 0.21242077510933857}


,model,ROC_AUC,PR_AUC,Brier,precision@5,recall@5,lift@5,precision@10,recall@10,lift@10,precision@20,recall@20,lift@20,precision@30,recall@30,lift@30
0,FLAML AutoML,0.7629,0.5397,0.1152,0.88,0.2472,4.9438,0.62,0.3483,3.4831,0.4433,0.4981,2.4906,0.3578,0.603,2.01


## 20. Extended XAI: PDP, ICE, SHAP dependence and surrogate model

No single explanation method is sufficient. I use several complementary views:

1. **Permutation importance** — model-agnostic global importance.
2. **SHAP** — local additive contributions and global importance.
3. **PDP** — average marginal relationship between a feature and prediction.
4. **ICE** — individual conditional expectation, showing heterogeneity hidden by PDP averages.
5. **Surrogate tree** — an intentionally simple approximation of the black-box model, useful for communicating broad decision logic.

The methods answer different questions and should not be interpreted as causal effects.

In [28]:

# Choose high-value continuous behavioural features that are available in the final model.
pdp_features = [f for f in [
    "salary", "age", "sessions", "total_session_min", "avg_session_min",
    "active_days", "recency_days", "number_of_previous_products"
] if f in X_test.columns]

# PDP: average model response as one feature varies.
if pdp_features:
    PartialDependenceDisplay.from_estimator(
        best_model, X_test, features=pdp_features[:4],
        kind="average", grid_resolution=30
    )
    plt.suptitle("Partial dependence — tuned XGBoost", y=1.02)
    plt.tight_layout()
    plt.show()

# ICE for two particularly interpretable behavioural variables.
ice_features = [f for f in ["sessions", "recency_days", "salary", "active_days"] if f in X_test.columns][:2]
if ice_features:
    PartialDependenceDisplay.from_estimator(
        best_model, X_test.sample(min(500, len(X_test)), random_state=RANDOM_STATE),
        features=ice_features, kind="both", subsample=100, random_state=RANDOM_STATE,
        grid_resolution=25
    )
    plt.suptitle("PDP + ICE — individual heterogeneity", y=1.02)
    plt.tight_layout()
    plt.show()

# SHAP dependence plots: feature value vs contribution.
for feature in [f for f in ["sessions", "recency_days", "salary"] if f in X_test.columns][:2]:
    idx = list(X_test.columns).index(feature)
    # For tree SHAP the transformed feature has the same numeric feature order before one-hot terms.
    transformed_name = f"num__{feature}"
    if transformed_name in feature_names:
        shap.dependence_plot(
            transformed_name, shap_values, X_test_transformed,
            feature_names=feature_names, show=False
        )
        plt.title(f"SHAP dependence — {feature}")
        plt.tight_layout()
        plt.show()

In [29]:
# Local explanations for a few individual users.
# This is useful for a marketing analyst asking: "Why did this customer rank highly?"
local_n = min(3, len(X_test))
local_idx = np.argsort(-best_proba)[:local_n]

for row_idx in local_idx:
    row = X_test.iloc[[row_idx]]
    transformed_row = best_model.named_steps["prep"].transform(row)
    local_shap = explainer.shap_values(transformed_row)[0]
    local_contrib = pd.Series(local_shap, index=feature_names).sort_values(key=np.abs, ascending=False)
    user_id_for_row = data.loc[row.index[0], "user_id"]
    print(f"\nUser: {user_id_for_row} | predicted probability={best_proba[row_idx]:.3f}")
    display(local_contrib.head(10).to_frame("SHAP contribution").round(4))


User: 1321 | predicted probability=0.980


,SHAP contribution
num__age,2.2153
num__salary,0.8415
num__avg_session_min,0.7422
num__gender,0.5458
num__sessions,0.2021
num__valid_session_dates,0.1799
num__active_days,0.1421
num__max_session_min,0.1147
num__flat_flg,0.1129
num__recency_days,0.1113



User: 4486 | predicted probability=0.963


,SHAP contribution
num__age,2.0038
num__gender,0.5746
num__sessions,0.5069
num__avg_session_min,0.4866
num__valid_session_dates,0.3476
num__active_days,0.3092
num__salary,0.1926
num__mean_lon,0.1366
num__mean_lat,0.1354
num__evening_share,0.1296



User: 5313 | predicted probability=0.952


,SHAP contribution
num__age,2.0835
num__avg_session_min,0.6218
num__gender,0.5453
num__salary,0.3823
num__max_session_min,0.3671
num__mean_lon,0.1417
num__recency_days,-0.1319
num__valid_session_dates,0.1250
num__mean_lat,0.1193
num__flat_flg,0.0988


In [30]:
# Simple surrogate decision tree.
# It is NOT the production model; it approximates the black-box ranking to communicate high-level rules.
X_train_pred = best_model.predict_proba(X_train)[:, 1]
surrogate = DecisionTreeClassifier(max_depth=3, min_samples_leaf=100, random_state=RANDOM_STATE)
surrogate.fit(X_train[numeric_features].fillna(X_train[numeric_features].median()), X_train_pred > np.median(X_train_pred))

fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    surrogate,
    feature_names=numeric_features,
    class_names=["lower model score", "higher model score"],
    filled=True,
    rounded=True,
    ax=ax
)
ax.set_title("Surrogate decision tree — approximation of model ranking")
plt.tight_layout()
plt.show()

## 21. External geo-economic enrichment — challenger extension

The original model deliberately uses only information supplied by the company. As a separate extension, we enrich users with a regional economic context variable from **Statistics Poland (GUS) Local Data Bank (BDL)**. BDL provides data for voivodeships and other territorial levels through a REST API. citeturn1search0turn1search1

The selected indicator is **GDP per capita at current prices (PKB na 1 mieszkańca)**, BDL variable `3501`, identified in public documentation referencing GUS BDL. citeturn2search24

Important modelling caveat: this is an **area-level contextual variable**, not the user's income. It should be treated as contextual enrichment, and the year must be chosen so that it was available before the prediction/campaign date. The notebook therefore treats it as a challenger model rather than silently changing the original model.

In [31]:
# Map our exploratory capital proxy to its voivodeship.
capital_to_voivodeship = {
    "Warsaw":"Mazowieckie", "Krakow":"Malopolskie", "Lodz":"Lodzkie",
    "Wroclaw":"Dolnoslaskie", "Poznan":"Wielkopolskie", "Gdansk":"Pomorskie",
    "Szczecin":"Zachodniopomorskie", "Bydgoszcz":"Kujawsko-Pomorskie",
    "Lublin":"Lubelskie", "Katowice":"Slaskie", "Bialystok":"Podlaskie",
    "Gorzow Wielkopolski":"Lubuskie", "Zielona Gora":"Lubuskie",
    "Rzeszow":"Podkarpackie", "Kielce":"Swietokrzyskie", "Olsztyn":"Warminsko-Mazurskie"
}

user_region = user_geo[["region_proxy"]].copy()
user_region["voivodeship"] = user_region["region_proxy"].map(capital_to_voivodeship)

# GUS BDL API download. We use a try/except so the core analysis remains reproducible even
# in a restricted/offline environment. The API documentation confirms JSON REST access.

def fetch_bdl_gdp_per_capita(year=2022):
    url = f"https://bdl.stat.gov.pl/api/v1/data/by-variable/3501?format=json&unit-level=2&year={year}&page-size=100"
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    payload = r.json()
    rows = []
    for item in payload.get("results", payload.get("Results", [])):
        unit = item.get("unit", {})
        name = unit.get("name") or item.get("name")
        for value in item.get("values", item.get("Values", [])):
            rows.append({"voivodeship": name, "gdp_per_capita": value.get("val", value.get("value")), "year": value.get("year", year)})
    return pd.DataFrame(rows)

try:
    gdp_regional = fetch_bdl_gdp_per_capita(year=2022)
    print("Downloaded GUS BDL observations:", gdp_regional.shape)
    display(gdp_regional.head(20))
except Exception as exc:
    gdp_regional = pd.DataFrame()
    print("GUS BDL download unavailable in this environment:", repr(exc))
    print("The extension remains fully reproducible when internet/API access is enabled.")

Downloaded GUS BDL observations: (16, 3)


,voivodeship,gdp_per_capita,year
0,MAŁOPOLSKIE,1391,2022
1,ŚLĄSKIE,1536,2022
2,LUBUSKIE,0,2022
3,WIELKOPOLSKIE,496,2022
4,ZACHODNIOPOMORSKIE,50,2022
5,DOLNOŚLĄSKIE,0,2022
6,OPOLSKIE,0,2022
7,KUJAWSKO-POMORSKIE,0,2022
8,POMORSKIE,0,2022
9,WARMIŃSKO-MAZURSKIE,507,2022


In [32]:
# Build the enriched modelling table only if GUS data were retrieved.
if not gdp_regional.empty:
    # Normalise Polish names to make the merge robust to diacritics/spacing differences.
    def norm_region(s):
        return (str(s).lower()
                .replace("ą","a").replace("ć","c").replace("ę","e").replace("ł","l")
                .replace("ń","n").replace("ó","o").replace("ś","s").replace("ź","z").replace("ż","z")
                .replace("-"," ").strip())

    gdp_regional["region_key"] = gdp_regional["voivodeship"].map(norm_region)
    user_region["region_key"] = user_region["voivodeship"].map(norm_region)
    enriched_map = user_region.reset_index(names="user_id").merge(
        gdp_regional[["region_key","gdp_per_capita"]], on="region_key", how="left"
    ).set_index("user_id")
    enriched_map = enriched_map[~enriched_map.index.duplicated(keep="first")]

    enriched = data.drop(columns=["segment"]).copy()
    enriched = enriched.merge(
        enriched_map[["voivodeship","gdp_per_capita"]],
        left_on="user_id", right_index=True, how="left"
    )

    # A missing value is imputed inside the challenger pipeline, just like other numeric features.
    X_geo = enriched.drop(columns=[target, "user_id"])
    y_geo = enriched[target].astype(int)
    categorical_geo = ["education", "voivodeship"]
    numeric_geo = [c for c in X_geo.columns if c not in categorical_geo]

    geo_preprocessor = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), numeric_geo),
        ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_geo),
    ])
    Xg_train, Xg_test, yg_train, yg_test = train_test_split(
        X_geo, y_geo, test_size=.25, stratify=y_geo, random_state=RANDOM_STATE
    )

    geo_xgb = Pipeline([
        ("prep", geo_preprocessor),
        ("model", XGBClassifier(
            n_estimators=search.best_params_["model__n_estimators"],
            max_depth=search.best_params_["model__max_depth"],
            learning_rate=search.best_params_["model__learning_rate"],
            subsample=search.best_params_["model__subsample"],
            colsample_bytree=search.best_params_["model__colsample_bytree"],
            min_child_weight=search.best_params_["model__min_child_weight"],
            reg_lambda=search.best_params_["model__reg_lambda"],
            reg_alpha=search.best_params_["model__reg_alpha"],
            eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
    geo_xgb.fit(Xg_train, yg_train)
    geo_proba = geo_xgb.predict_proba(Xg_test)[:,1]

    geo_metrics = {
        "model":"Tuned XGBoost + regional GDP per capita",
        "ROC_AUC":roc_auc_score(yg_test, geo_proba),
        "PR_AUC":average_precision_score(yg_test, geo_proba),
        "Brier":brier_score_loss(yg_test, geo_proba),
    }
    geo_metrics.update(campaign_metrics(yg_test, geo_proba))
    display(pd.DataFrame([geo_metrics]).round(4))

    # The key question is incremental value, not whether GDP itself is correlated with outcome.
    base_geo_comparison = pd.DataFrame([
        {"model":"Original tuned XGBoost", "PR_AUC":average_precision_score(yg_test, best_model.predict_proba(Xg_test.drop(columns=["gdp_per_capita","voivodeship"], errors="ignore"))[:,1])},
        {"model":"+ regional GDP per capita", "PR_AUC":geo_metrics["PR_AUC"]}
    ])
    display(base_geo_comparison.round(4))
else:
    print("Skipping geo-economic challenger because GUS BDL data were not downloaded.")

,model,ROC_AUC,PR_AUC,Brier,precision@5,recall@5,lift@5,precision@10,recall@10,lift@10,precision@20,recall@20,lift@20,precision@30,recall@30,lift@30
0,Tuned XGBoost + regional GDP per capita,0.7595,0.5344,0.1156,0.84,0.236,4.7191,0.6067,0.3408,3.4082,0.4333,0.4869,2.4345,0.3689,0.6217,2.0724


,model,PR_AUC
0,Original tuned XGBoost,0.5378
1,+ regional GDP per capita,0.5344


## 22. MLflow experiment tracking — local

MLflow is part of the modelling workflow, not an optional add-on. For this individual exercise no Tracking Server is necessary: run metadata are stored in **SQLite** and artifacts on the **local filesystem**.

The notebook logs candidate model parameters, metrics, business lift metrics and model artifacts. In a team environment the same experiment can later be moved to a shared MLflow Tracking Server with remote artifact storage.


In [33]:
# MLflow is mandatory and configured through the reusable MLflowTracker class.
tracker = MLflowTracker(CONFIG)

mlflow_runs = []
for model_name, model_obj, pred in [
    ("baseline_xgboost", models["XGBoost"], test_predictions["XGBoost"]),
    ("tuned_xgboost", best_model, best_proba),
    ("random_forest", models["Random forest"], test_predictions["Random forest"]),
    ("logistic_regression", models["Logistic regression"], test_predictions["Logistic regression"]),
    ("out_of_box_hist_gradient_boosting", oob_model, oob_proba),
    ("neural_network_mlp", mlp_model, mlp_proba),
]:
    run_id, metrics = tracker.log_classifier(model_name, model_obj, y_test, pred)
    mlflow_runs.append({"run_name": model_name, "run_id": run_id, **metrics})

mlflow_runs_df = pd.DataFrame(mlflow_runs).sort_values("pr_auc", ascending=False)
display(mlflow_runs_df.round(4))
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Local artifacts:", tracker.artifacts)

with mlflow.start_run(run_name="flaml_automl"):
    mlflow.log_metrics({"roc_auc": roc_auc_score(y_test, automl_proba), "pr_auc": average_precision_score(y_test, automl_proba), "brier": brier_score_loss(y_test, automl_proba)})
    mlflow.log_params({"best_estimator": str(automl.best_estimator), "best_config": str(automl.best_config), "time_budget": 120})


,run_name,run_id,roc_auc,pr_auc,brier,precision_5,recall_5,lift_5,precision_10,recall_10,lift_10,precision_20,recall_20,lift_20,precision_30,recall_30,lift_30
0,baseline_xgboost,02744a45c359491fa2ec6714c6d3103a,0.7623,0.5390,0.1152,0.8400,0.2360,4.7191,0.6133,0.3446,3.4457,0.4400,0.4944,2.4719,0.3578,0.6030,2.0100
1,tuned_xgboost,ff7a3852cafb478fb51ad5c2a014d95a,0.7575,0.5378,0.1153,0.8533,0.2397,4.7940,0.6000,0.3371,3.3708,0.4333,0.4869,2.4345,0.3667,0.6180,2.0599
4,out_of_box_hist_gradient_boosting,02c7570f3d0f4574ab52efc29b9cc71c,0.7592,0.5296,0.1164,0.8400,0.2360,4.7191,0.6133,0.3446,3.4457,0.4200,0.4719,2.3596,0.3667,0.6180,2.0599
2,random_forest,dfc2f49a7b134c2a80ee9e6d9d5b42f3,0.7533,0.5232,0.1310,0.8267,0.2322,4.6442,0.5867,0.3296,3.2959,0.4367,0.4906,2.4532,0.3644,0.6142,2.0474
5,neural_network_mlp,c2df4d60bdd74987b11d605c639a31e8,0.7352,0.5095,0.1202,0.8133,0.2285,4.5693,0.5867,0.3296,3.2959,0.4267,0.4794,2.3970,0.3467,0.5843,1.9476
3,logistic_regression,a631166f01e44a36bc1dfc81ef7696dd,0.7368,0.5001,0.2012,0.7733,0.2172,4.3446,0.5733,0.3221,3.2210,0.4233,0.4757,2.3783,0.3556,0.5993,1.9975


MLflow tracking URI: sqlite:////home/maciek/Desktop/inpost_recru/mlflow_runs/mlflow.db
Local artifacts: /home/maciek/Desktop/inpost_recru/mlflow_runs/artifacts


In [34]:
# Cross-validation sanity check for the two XGBoost candidates.
# We should not select a model by repeatedly looking at the test set. The test set is kept as a
# final audit; model selection is primarily based on cross-validation within the training data.

baseline_xgb = models["XGBoost"]
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline_cv = cross_val_score(
    baseline_xgb, X_train, y_train, cv=cv5, scoring="average_precision", n_jobs=-1
)
tuned_cv = cross_val_score(
    best_model, X_train, y_train, cv=cv5, scoring="average_precision", n_jobs=-1
)

cv_comparison = pd.DataFrame({
    "model":["Baseline XGBoost", "Tuned XGBoost"],
    "CV_PR_AUC_mean":[baseline_cv.mean(), tuned_cv.mean()],
    "CV_PR_AUC_std":[baseline_cv.std(), tuned_cv.std()],
})
display(cv_comparison.round(4))

print("Important: the untouched test set is used only for the final audit. If CV favours the tuned model but the test result is slightly lower, that is normal sampling variance. We should not repeatedly retune against the test set.")


,model,CV_PR_AUC_mean,CV_PR_AUC_std
0,Baseline XGBoost,0.5165,0.0329
1,Tuned XGBoost,0.5167,0.0308


Important: the untouched test set is used only for the final audit. If CV favours the tuned model but the test result is slightly lower, that is normal sampling variance. We should not repeatedly retune against the test set.


In [35]:
# Repeated CV reduces the chance of over-interpreting one random fold split.
repeated_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
base_rep = cross_val_score(models["XGBoost"], X_train, y_train, cv=repeated_cv, scoring="average_precision", n_jobs=-1)
tuned_rep = cross_val_score(best_model, X_train, y_train, cv=repeated_cv, scoring="average_precision", n_jobs=-1)
repeated_cv_summary = pd.DataFrame({
    "model": ["Baseline XGBoost", "Tuned XGBoost"],
    "mean_PR_AUC": [base_rep.mean(), tuned_rep.mean()],
    "std_PR_AUC": [base_rep.std(), tuned_rep.std()],
})
display(repeated_cv_summary.round(4))
print("Mean paired difference (tuned - baseline):", round((tuned_rep - base_rep).mean(), 4))


,model,mean_PR_AUC,std_PR_AUC
0,Baseline XGBoost,0.5229,0.0326
1,Tuned XGBoost,0.5172,0.0313


Mean paired difference (tuned - baseline): -0.0057


## 23. Final model-selection recommendation

**Select baseline (untuned) XGBoost for a controlled pilot.** Repeated stratified 5-fold CV over 3 repeats gives mean PR-AUC **0.5229 ± 0.0326**, compared with **0.5172 ± 0.0313** for tuned XGBoost (paired mean difference, tuned minus baseline: **−0.0057**). We choose the baseline because tuning did not improve average ranking performance and adds complexity.

On the untouched 25% holdout, baseline XGBoost achieves ROC-AUC **0.7623**, PR-AUC **0.5390**, and Brier score **0.1152**. The top-scored 10% has **61.3% precision**, **34.5% recall**, and **3.45× lift**. These metrics support a ranking pilot, not a claim of causal campaign impact.

The MLP, out-of-the-box benchmark and AutoML are useful challengers because they test whether neural modelling, minimal effort or automated search materially improves the ranking problem.

The geo-economic model should only be promoted if external regional context provides stable incremental lift and was available before the prediction date.


## 23A. Executive findings and decision log

### Business
- The dataset contains **6,000 users**, with a historical premium rate of roughly **17.8%**.
- The data show where observed users are concentrated, but **not true geographic penetration**; population denominators are required before calling a region “underpenetrated”.
- Behavioural activity, recency and existing customer information are natural candidates for campaign prioritisation.

### Modelling
- **PR-AUC and lift@10%** are more decision-relevant than accuracy because campaign capacity is limited and the positive class is a minority.
- Tree-based models are strong candidates for this tabular problem; the MLP is a challenger, not an automatic improvement.
- Hyperparameter tuning must earn its complexity through stable cross-validation improvement. If the gain is small or unstable, prefer the simpler XGBoost model.

### MLOps
- The notebook separates configuration, validation, feature engineering, model construction and MLflow tracking into reusable classes.
- In production these classes should become package modules with unit tests, CI/CD, versioned data/features and a model registry.

### Most important next step
The current model is a **propensity model**, not an incremental-impact model. A randomized campaign experiment followed by **uplift/treatment-effect modelling** is the strongest path to optimising incremental ARPU.


In [36]:
# Compact final comparison for the presentation.
comparison_columns = [
    "model", "ROC_AUC", "PR_AUC", "Brier", "precision@10", "lift@10"
]

final_candidates = (
    pd.DataFrame([oob_metrics, mlp_metrics, automl_metrics])[comparison_columns]
    .sort_values("PR_AUC", ascending=False)
)
display(final_candidates.round(4))

,model,ROC_AUC,PR_AUC,Brier,precision@10,lift@10
2,FLAML AutoML,0.7629,0.5397,0.1152,0.6200,3.4831
0,Out-of-box HistGradientBoosting,0.7592,0.5296,0.1164,0.6133,3.4457
1,Neural network (MLP),0.7352,0.5095,0.1202,0.5867,3.2959


## 24. MLOps quality gates

These lightweight checks would become automated tests in CI in a production repository.


In [37]:
assert users["user_id"].is_unique
assert not set(X_train.index).intersection(set(X_test.index))
assert target not in X_train.columns and target not in X_test.columns
assert len(y_train) + len(y_test) == len(data)
assert 0 < y_train.mean() < 1 and 0 < y_test.mean() < 1
assert isinstance(best_model, Pipeline)
assert "prep" in best_model.named_steps and "model" in best_model.named_steps
assert final_metrics["PR_AUC"] >= target_share
print("All MLOps quality gates passed.")


All MLOps quality gates passed.
